# Experiment 04 — Per-family models vs the global model

This is a **standalone experiment** — it does not modify the main pipeline (notebooks 01–03). It answers one question: do 33 separate LightGBM models (one per product family) beat the single global model, and can a hybrid/blend of the two improve the public score (current submission: **0.39109**)?

> 🇷🇺 Отдельный эксперимент, основной пайплайн (01–03) не меняется. Вопрос: бьют ли 33 модели по категориям одну глобальную, и улучшает ли гибрид/бленд публичный скор (текущий сабмит: **0.39109**).

**Design / Дизайн:**

1. **Rolling-origin validation on 3 windows** of 16 days each (mirroring the test horizon): `W1` 2017-06-15…06-30, `W2` 2017-07-15…07-30, `W3` 2017-07-31…08-15. A scheme is accepted only if it does not lose on *any* window — protection against tuning to a single window.
2. **Honest training cutoffs.** For every window all models are trained strictly on `date < window_start`. ⚠️ This experiment is where the leakage in notebook 03 was originally caught: `model_2016`/`model_2017` used to be trained on the *full* train (no upper cutoff), i.e. they saw the validation window 07-31…08-15 during training — a big part of the 0.293 (local) vs 0.391 (Kaggle) gap. Notebook 03 has since been fixed (separate `*_val` models trained before the window). The numbers below are honest by construction.
3. **Same iterative day-by-day forecasting** as notebook 03: window sales are hidden, lag/rolling features are recomputed from the model's own predictions. Implementation differences vs 03 (both apply equally to all schemes and to the final test forecast, so the comparison and the deployment stay consistent): hidden sales are `NaN` (as in the test region) instead of `0.0`, and `rolling_mean_364` is updated inside the loop from own predictions instead of being frozen before it.
4. **Schemes compared** (all LightGBM, identical hyperparameters from notebook 03 — one variable at a time):
   - `global_2016` / `global_2017` — one model on all families, trained from 2016-01-01 / 2017-01-01;
   - `per_family` — 33 models, one per family, trained from 2016-01-01;
   - `blend_*` — weighted averages of the above inside the iterative loop;
   - `hybrid_*` — per-family model selection: a family uses an alternative scheme only if it beats the default on **all 3 windows** (stitched from the base runs — valid because no feature crosses family boundaries, so per-family trajectories are independent).
5. **Decision gate.** The final submission scheme must be ≤ `global_2017` (the honest proxy of the current submission, which used `w_2017 = 1.0`) on every window; among gated schemes the best mean wins. Models for the test forecast are then retrained on the full train (`date < 2017-08-16` — no leak: test sales are unknown by definition).

> 🇷🇺 Кратко: 3 окна валидации, обучение строго до начала окна (здесь была поймана утечка ноутбука 03 — финальные модели видели валидационное окно; в 03 теперь исправлено), тот же итеративный прогноз, идентичные гиперпараметры; гибрид собирается по правилу «категория переключается на альтернативную модель, только если та лучше на всех 3 окнах»; финальная схема обязана не проигрывать `global_2017` ни на одном окне.

In [1]:
import time

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib

In [2]:
_t0 = time.time()

# SMOKE=1 runs the whole notebook with tiny models to verify the code path
# SMOKE=1 прогоняет весь ноутбук с крошечными моделями для проверки кода
SMOKE = os.environ.get("EXP04_SMOKE") == "1"
N_EST = 60 if SMOKE else 4155  # 4155 = best_iteration_ + 100 from notebook 03

MODELS_DIR = "../models/exp04" + ("_smoke" if SMOKE else "")
ART_DIR    = "../artifacts/exp04" + ("_smoke" if SMOKE else "")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(ART_DIR, exist_ok=True)

# Same hyperparameters as the final models in notebook 03 / Те же гиперпараметры, что у финальных моделей в 03
LGBM_PARAMS = dict(
    n_estimators=N_EST,
    learning_rate=0.01,
    num_leaves=255,
    min_child_samples=30,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

FEATURES_V2 = [
    "day_of_week", "month", "year", "is_weekend",
    "day_of_year", "week_of_year", "date_index",
    "sin_day", "cos_day", "sin_week", "cos_week",
    "lag_1", "lag_2", "lag_3", "lag_4", "lag_5", "lag_6",
    "lag_7", "lag_14", "lag_21", "lag_28", "lag_42", "lag_56",
    "lag_364", "lag_365",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28", "rolling_mean_364",
    "dcoilwtico", "dcoilwtico_ma7", "dcoilwtico_ma28",
    "onpromotion", "onpromotion_ma7", "onpromotion_ma28",
    *[f"transactions_lag_{l}" for l in range(16, 24)],
    "is_holiday_national", "day_before_holiday",
    "is_black_friday", "is_cyber_monday", "is_terremoto",
    "is_navidad", "is_dia_madre", "is_futbol",
    "is_dia_trabajo", "is_primer_dia", "is_dia_difuntos", "work_day",
    "store_nbr", "store_type", "cluster",
    "family",
]
CAT_FEATURES = ["store_nbr", "store_type", "cluster", "family", "day_of_week", "month"]

WINDOWS = [
    ("W1", pd.Timestamp("2017-06-15"), pd.Timestamp("2017-06-30")),
    ("W2", pd.Timestamp("2017-07-15"), pd.Timestamp("2017-07-30")),
    ("W3", pd.Timestamp("2017-07-31"), pd.Timestamp("2017-08-15")),
]
WNAMES = [w for w, _, _ in WINDOWS]
TEST_START, TEST_END = pd.Timestamp("2017-08-16"), pd.Timestamp("2017-08-31")

print(f"SMOKE={SMOKE}, n_estimators={N_EST}")

SMOKE=False, n_estimators=4155


In [3]:
# Load features and reshape sales into a (dates x series) matrix — the data is rectangular,
# which allows lag features to be computed by simple row slicing inside the iterative loop.
# Загружаем признаки; продажи укладываются в матрицу (даты x ряды) — данные прямоугольные,
# и лаги внутри итеративного цикла считаются простым срезом строк.
DF = (
    pd.read_parquet("../artifacts/features.parquet")
    .sort_values(["date", "store_nbr", "family"])
    .reset_index(drop=True)
)

DATES = np.sort(DF["date"].unique())
N_DATES = len(DATES)
N_SERIES = DF.groupby("date").size().iloc[0]
assert len(DF) == N_DATES * N_SERIES, "calendar is not rectangular"
DATE_IDX = {d: i for i, d in enumerate(DATES)}

# The (store, family) order must be identical inside every date block / Порядок (магазин, категория) должен совпадать в каждом дневном блоке
ref = DF.iloc[:N_SERIES][["store_nbr", "family"]].reset_index(drop=True)
chk = DF.iloc[1000 * N_SERIES:1001 * N_SERIES][["store_nbr", "family"]].reset_index(drop=True)
assert ref.equals(chk), "series order differs between dates"

FAMILIES = sorted(DF["family"].astype(str).unique())
FAM_ARR = ref["family"].astype(str).values            # family of each of the 1782 columns
SALES_TRUE = DF["sales"].to_numpy(dtype=float).reshape(N_DATES, N_SERIES)  # test rows are NaN

print(f"{N_DATES} dates x {N_SERIES} series, {len(FAMILIES)} families")
print(f"range: {pd.Timestamp(DATES[0]).date()} .. {pd.Timestamp(DATES[-1]).date()}")

1704 dates x 1782 series, 33 families
range: 2013-01-01 .. 2017-08-31


In [4]:
# Engine: training, dynamic features, iterative forecast, predictors
# Движок: обучение, динамические признаки, итеративный прогноз, предикторы
LAGS  = [1, 2, 3, 4, 5, 6, 7, 14, 21, 28, 42, 56, 364, 365]
ROLLS = [7, 14, 28]


def rmsle_mat(P, Y):
    return float(np.sqrt(np.mean((np.log1p(np.clip(P, 0, None)) - np.log1p(Y)) ** 2)))


def dynamic_features(sales_mat, i):
    """Lag/rolling features for date index i from the current state of sales_mat.
    Definitions mirror notebook 02: lag_k = sales(t-k); rolling = mean over the previous w days
    (shift(1), min_periods=1; rolling_mean_364 with min_periods=30)."""
    feat = {}
    for k in LAGS:
        feat[f"lag_{k}"] = sales_mat[i - k]
    for w in ROLLS:
        feat[f"rolling_mean_{w}"] = np.nanmean(sales_mat[i - w:i], axis=0)
    win = sales_mat[i - 364:i]
    cnt = (~np.isnan(win)).sum(axis=0)
    rm = np.full(N_SERIES, np.nan)
    ok = cnt >= 30
    if ok.any():
        rm[ok] = np.nanmean(win[:, ok], axis=0)
    feat["rolling_mean_364"] = rm
    return feat


def x_for_day(i, sales_mat):
    X = DF.iloc[i * N_SERIES:(i + 1) * N_SERIES][FEATURES_V2].copy()
    for c, v in dynamic_features(sales_mat, i).items():
        X[c] = v
    for col in CAT_FEATURES:
        X[col] = X[col].astype("category")
    return X


def zero_rule_mask(first_idx):
    """Series with zero sales over the 21 days before the window are forecast as 0 (as in 03)."""
    return np.nansum(SALES_TRUE[first_idx - 21:first_idx], axis=0) == 0


def run_iterative(predict_day, window_dates, zmask):
    """Honest day-by-day forecast: window sales hidden as NaN, features recomputed from own predictions."""
    sm = SALES_TRUE.copy()
    idxs = [DATE_IDX[d] for d in window_dates]
    sm[idxs, :] = np.nan
    P = np.empty((len(idxs), N_SERIES))
    for j, i in enumerate(idxs):
        p = np.clip(predict_day(x_for_day(i, sm)), 0, None)
        p[zmask] = 0.0
        sm[i] = p
        P[j] = p
    return P


def fit_lgbm(sub):
    X = sub[FEATURES_V2].copy()
    y = np.log1p(sub["sales"])
    for col in CAT_FEATURES:
        X[col] = X[col].astype("category")
    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(X, y)
    return m


def train_global(train_from, cutoff, tag):
    path = os.path.join(MODELS_DIR, f"global_{tag}.joblib")
    if os.path.exists(path):
        return joblib.load(path)
    sub = DF[(DF["date"] >= train_from) & (DF["date"] < cutoff) & DF["sales"].notna()]
    t = time.time()
    m = fit_lgbm(sub)
    joblib.dump(m, path)
    print(f"  trained global_{tag}: {len(sub):,} rows, {time.time() - t:.0f}s")
    return m


def train_per_family(train_from, cutoff, tag):
    base = DF[(DF["date"] >= train_from) & (DF["date"] < cutoff) & DF["sales"].notna()]
    models, t = {}, time.time()
    for fam in FAMILIES:
        slug = fam.replace("/", "_").replace(" ", "_")
        path = os.path.join(MODELS_DIR, f"perfam_{tag}_{slug}.joblib")
        if os.path.exists(path):
            models[fam] = joblib.load(path)
        else:
            models[fam] = fit_lgbm(base[base["family"] == fam])
            joblib.dump(models[fam], path)
    print(f"  per-family x{len(FAMILIES)} ({tag}): {time.time() - t:.0f}s")
    return models


def predictor_global(model):
    def f(X):
        return np.expm1(model.predict(X))
    return f


def predictor_per_family(models):
    def f(X):
        fam = X["family"].astype(str).values
        out = np.empty(len(X))
        for name, m in models.items():
            mask = fam == name
            out[mask] = np.expm1(m.predict(X[mask]))
        return out
    return f


def predictor_blend(f1, f2, w1):
    def f(X):
        return w1 * f1(X) + (1.0 - w1) * f2(X)
    return f

In [5]:
# Sanity check: matrix features must reproduce the parquet features of notebook 02 exactly
# Проверка корректности: матричные признаки должны в точности совпадать с parquet-признаками из 02
i = DATE_IDX[pd.Timestamp("2017-05-10")]
feat = dynamic_features(SALES_TRUE, i)
block = DF.iloc[i * N_SERIES:(i + 1) * N_SERIES]
for c in ["lag_1", "lag_7", "lag_364", "rolling_mean_7", "rolling_mean_28", "rolling_mean_364"]:
    a = np.asarray(feat[c], dtype=float)
    b = block[c].to_numpy(dtype=float)
    assert np.allclose(a, b, equal_nan=True, atol=1e-6), f"mismatch in {c}"
print("OK: matrix engine reproduces notebook 02 feature definitions")

OK: matrix engine reproduces notebook 02 feature definitions


In [6]:
# Main runs: per window — train with an honest cutoff, validate all schemes iteratively
# Основные прогоны: для каждого окна — обучение с честной отсечкой, итеративная валидация всех схем
RESULTS = {}            # (window, scheme) -> P (16 x N_SERIES)
rows, fam_rows = [], []

for wname, ws, we in WINDOWS:
    print(f"\n=== {wname}: {ws.date()} .. {we.date()} ===")
    wdates = [d for d in DATES if ws <= d <= we]
    assert len(wdates) == 16
    si = DATE_IDX[wdates[0]]
    zmask = zero_rule_mask(si)
    Y = SALES_TRUE[[DATE_IDX[d] for d in wdates]]

    g16 = train_global(pd.Timestamp("2016-01-01"), ws, f"2016_{wname}")
    g17 = train_global(pd.Timestamp("2017-01-01"), ws, f"2017_{wname}")
    pf  = train_per_family(pd.Timestamp("2016-01-01"), ws, wname)

    fg16, fg17, fpf = predictor_global(g16), predictor_global(g17), predictor_per_family(pf)
    schemes = {
        "global_2016":      fg16,
        "global_2017":      fg17,
        "per_family":       fpf,
        "blend_g16_g17_50": predictor_blend(fg16, fg17, 0.5),
        "blend_pf25_g16":   predictor_blend(fpf, fg16, 0.25),
        "blend_pf50_g16":   predictor_blend(fpf, fg16, 0.50),
        "blend_pf75_g16":   predictor_blend(fpf, fg16, 0.75),
        "blend_pf50_g17":   predictor_blend(fpf, fg17, 0.50),
    }

    for name, fn in schemes.items():
        cache = os.path.join(ART_DIR, f"pred_{wname}__{name}.npy")
        t = time.time()
        if os.path.exists(cache):
            P = np.load(cache)
        else:
            P = run_iterative(fn, wdates, zmask)
            np.save(cache, P)
        RESULTS[(wname, name)] = P
        sc = rmsle_mat(P, Y)
        rows.append({"window": wname, "scheme": name, "rmsle": sc})
        for fam in FAMILIES:
            m = FAM_ARR == fam
            fam_rows.append({"window": wname, "scheme": name, "family": fam,
                             "rmsle": rmsle_mat(P[:, m], Y[:, m])})
        print(f"  {name:18s} RMSLE={sc:.5f} ({time.time() - t:.0f}s)")

res = pd.DataFrame(rows)
famres = pd.DataFrame(fam_rows)
res.to_csv(os.path.join(ART_DIR, "results_base.csv"), index=False)
famres.to_csv(os.path.join(ART_DIR, "results_per_family.csv"), index=False)
print(f"\nElapsed: {(time.time() - _t0) / 60:.1f} min")


=== W1: 2017-06-15 .. 2017-06-30 ===
  per-family x33 (W1): 15s
  global_2016        RMSLE=0.38302 (0s)
  global_2017        RMSLE=0.39866 (0s)
  per_family         RMSLE=0.39154 (0s)
  blend_g16_g17_50   RMSLE=0.38765 (0s)
  blend_pf25_g16     RMSLE=0.38195 (0s)
  blend_pf50_g16     RMSLE=0.38323 (0s)
  blend_pf75_g16     RMSLE=0.38647 (0s)
  blend_pf50_g17     RMSLE=0.38737 (0s)

=== W2: 2017-07-15 .. 2017-07-30 ===
  per-family x33 (W2): 17s
  global_2016        RMSLE=0.38801 (0s)
  global_2017        RMSLE=0.38940 (0s)
  per_family         RMSLE=0.39413 (0s)
  blend_g16_g17_50   RMSLE=0.38700 (0s)
  blend_pf25_g16     RMSLE=0.38681 (0s)
  blend_pf50_g16     RMSLE=0.38764 (0s)
  blend_pf75_g16     RMSLE=0.38998 (0s)
  blend_pf50_g17     RMSLE=0.38766 (0s)

=== W3: 2017-07-31 .. 2017-08-15 ===
  per-family x33 (W3): 15s
  global_2016        RMSLE=0.40490 (0s)
  global_2017        RMSLE=0.41564 (0s)
  per_family         RMSLE=0.40869 (0s)
  blend_g16_g17_50   RMSLE=0.40803 (0s)
  ble

In [7]:
# Stitched hybrids: per-family scheme selection on validation
# A family switches from the default scheme to a candidate only if the candidate beats the default
# on ALL 3 windows (this is exact, not an approximation: no feature crosses family boundaries,
# so each family's iterative trajectory is independent of the others).
# Сшитые гибриды: выбор схемы по каждой категории на валидации
# Категория переключается с дефолтной схемы на кандидата, только если кандидат лучше на ВСЕХ 3 окнах
# (это точно, а не приближение: ни один признак не пересекает границы категорий,
# поэтому итеративная траектория каждой категории независима от остальных).
R = famres.set_index(["window", "scheme", "family"])["rmsle"]


def pick_sources(default, candidates):
    sources = {}
    for fam in FAMILIES:
        beats = [c for c in candidates
                 if all(R[(w, c, fam)] < R[(w, default, fam)] for w in WNAMES)]
        if beats:
            sources[fam] = min(beats, key=lambda c: np.mean([R[(w, c, fam)] for w in WNAMES]))
        else:
            sources[fam] = default
    return sources


def stitch(sources):
    out = {}
    for w in WNAMES:
        P = np.empty((16, N_SERIES))
        for fam, src in sources.items():
            m = FAM_ARR == fam
            P[:, m] = RESULTS[(w, src)][:, m]
        out[w] = P
    return out


# Default = the better of the two global models by mean RMSLE / Дефолт = лучшая из двух глобальных по среднему RMSLE
mean_rmsle = res.groupby("scheme")["rmsle"].mean()
DEFAULT_GLOBAL = "global_2016" if mean_rmsle["global_2016"] <= mean_rmsle["global_2017"] else "global_2017"

HYBRIDS = {
    "hybrid_pf":   pick_sources(DEFAULT_GLOBAL, ["per_family"]),
    "hybrid_best": pick_sources(DEFAULT_GLOBAL,
                                [s for s in mean_rmsle.index if s != DEFAULT_GLOBAL]),
}

hyb_rows = []
for hname, sources in HYBRIDS.items():
    stitched = stitch(sources)
    for wname, ws, we in WINDOWS:
        wdates = [d for d in DATES if ws <= d <= we]
        Y = SALES_TRUE[[DATE_IDX[d] for d in wdates]]
        RESULTS[(wname, hname)] = stitched[wname]
        hyb_rows.append({"window": wname, "scheme": hname,
                         "rmsle": rmsle_mat(stitched[wname], Y)})
    switched = {f: s for f, s in sources.items() if s != DEFAULT_GLOBAL}
    print(f"{hname}: default={DEFAULT_GLOBAL}, {len(switched)} families switched")
    for f, s in switched.items():
        print(f"   {f} -> {s}")

res_all = pd.concat([res, pd.DataFrame(hyb_rows)], ignore_index=True)
res_all.to_csv(os.path.join(ART_DIR, "results_all.csv"), index=False)

hybrid_pf: default=global_2016, 2 families switched
   DAIRY -> per_family
   PERSONAL CARE -> per_family
hybrid_best: default=global_2016, 14 families switched
   AUTOMOTIVE -> blend_pf25_g16
   BEVERAGES -> blend_pf50_g16
   BOOKS -> global_2017
   BREAD/BAKERY -> blend_pf25_g16
   DAIRY -> blend_pf75_g16
   DELI -> blend_pf50_g16
   GROCERY I -> blend_pf25_g16
   HOME AND KITCHEN II -> blend_pf50_g16
   LIQUOR,WINE,BEER -> blend_pf25_g16
   MEATS -> blend_pf50_g16
   PERSONAL CARE -> blend_pf75_g16
   POULTRY -> blend_pf50_g17
   PRODUCE -> blend_pf50_g16
   SCHOOL AND OFFICE SUPPLIES -> blend_pf25_g16


In [8]:
# Summary table and decision gate
# The final scheme must not lose to global_2017 (the honest proxy of the current submission,
# which used w_2017 = 1.0) on ANY window; among gated schemes the best mean wins.
# Сводная таблица и решающее правило
# Финальная схема не должна проигрывать global_2017 (честный прокси текущего сабмита,
# где w_2017 = 1.0) ни на одном окне; среди прошедших гейт побеждает лучшее среднее.
piv = res_all.pivot(index="scheme", columns="window", values="rmsle")
piv["mean"] = piv[WNAMES].mean(axis=1)
piv = piv.sort_values("mean")
display(piv.round(5))

gate = piv[(piv[WNAMES] <= piv.loc["global_2017", WNAMES] + 1e-12).all(axis=1)]
FINAL_SCHEME = gate["mean"].idxmin() if len(gate) else "global_2017"
print(f"\nGate passed by: {list(gate.index)}")
print(f"FINAL_SCHEME = {FINAL_SCHEME}")
print(f"  mean RMSLE {piv.loc[FINAL_SCHEME, 'mean']:.5f} vs global_2017 {piv.loc['global_2017', 'mean']:.5f}")

window,W1,W2,W3,mean
scheme,,,,
hybrid_best,0.38205,0.38683,0.40231,0.39039
blend_pf25_g16,0.38195,0.38681,0.40261,0.39046
blend_pf50_g16,0.38323,0.38764,0.40242,0.39110
hybrid_pf,0.38289,0.38778,0.40469,0.39179
global_2016,0.38302,0.38801,0.40490,0.39198
blend_pf75_g16,0.38647,0.38998,0.40430,0.39358
blend_pf50_g17,0.38737,0.38766,0.40585,0.39363
blend_g16_g17_50,0.38765,0.38700,0.40803,0.39423
per_family,0.39154,0.39413,0.40869,0.39812



Gate passed by: ['hybrid_best', 'blend_pf25_g16', 'blend_pf50_g16', 'hybrid_pf', 'global_2016', 'blend_pf50_g17', 'blend_g16_g17_50', 'global_2017']
FINAL_SCHEME = hybrid_best
  mean RMSLE 0.39039 vs global_2017 0.40123


In [9]:
# Final: retrain on the full train (date < 2017-08-16) and forecast the test iteratively
# Финал: переобучение на всём train (date < 2017-08-16) и итеративный прогноз теста
CUTOFF = TEST_START  # honest by definition: test sales are unknown / честно по определению: продажи теста неизвестны

g16_full = train_global(pd.Timestamp("2016-01-01"), CUTOFF, "2016_FULL")
g17_full = train_global(pd.Timestamp("2017-01-01"), CUTOFF, "2017_FULL")
pf_full  = train_per_family(pd.Timestamp("2016-01-01"), CUTOFF, "FULL")

fg16, fg17, fpf = predictor_global(g16_full), predictor_global(g17_full), predictor_per_family(pf_full)
FULL_FNS = {
    "global_2016":      fg16,
    "global_2017":      fg17,
    "per_family":       fpf,
    "blend_g16_g17_50": predictor_blend(fg16, fg17, 0.5),
    "blend_pf25_g16":   predictor_blend(fpf, fg16, 0.25),
    "blend_pf50_g16":   predictor_blend(fpf, fg16, 0.50),
    "blend_pf75_g16":   predictor_blend(fpf, fg16, 0.75),
    "blend_pf50_g17":   predictor_blend(fpf, fg17, 0.50),
}

# For hybrids each family keeps the scheme chosen on validation / Для гибридов каждая категория сохраняет схему, выбранную на валидации
if FINAL_SCHEME in HYBRIDS:
    SOURCES = HYBRIDS[FINAL_SCHEME]
else:
    SOURCES = {fam: FINAL_SCHEME for fam in FAMILIES}


def predict_final(X):
    fam = X["family"].astype(str).values
    out = np.empty(len(X))
    cache = {s: FULL_FNS[s](X) for s in set(SOURCES.values())}
    for f_name in FAMILIES:
        m = fam == f_name
        out[m] = cache[SOURCES[f_name]][m]
    return out


test_dates = [d for d in DATES if TEST_START <= d <= TEST_END]
assert len(test_dates) == 16
zmask_test = zero_rule_mask(DATE_IDX[test_dates[0]])
print(f"Zero-rule: {int(zmask_test.sum())} series = 0")

cache_path = os.path.join(ART_DIR, f"pred_TEST__{FINAL_SCHEME}.npy")
if os.path.exists(cache_path):
    P_test = np.load(cache_path)
else:
    t = time.time()
    P_test = run_iterative(predict_final, test_dates, zmask_test)
    np.save(cache_path, P_test)
    print(f"test forecast: {time.time() - t:.0f}s")

for j, d in enumerate(test_dates):
    print(f"{pd.Timestamp(d).date()}: sum = {P_test[j].sum():,.0f}")

  per-family x33 (FULL): 14s
Zero-rule: 127 series = 0
2017-08-16: sum = 789,956
2017-08-17: sum = 646,263
2017-08-18: sum = 747,848
2017-08-19: sum = 819,155
2017-08-20: sum = 914,719
2017-08-21: sum = 754,835
2017-08-22: sum = 712,051
2017-08-23: sum = 731,587
2017-08-24: sum = 622,992
2017-08-25: sum = 736,777
2017-08-26: sum = 819,202
2017-08-27: sum = 910,473
2017-08-28: sum = 737,201
2017-08-29: sum = 711,179
2017-08-30: sum = 760,444
2017-08-31: sum = 661,099


In [10]:
# Build the submission / Собираем сабмишн
test = pd.read_csv("../data/test.csv", parse_dates=["date"])

long = pd.concat(
    [pd.DataFrame({"date": d, "store_nbr": ref["store_nbr"].values,
                   "family": ref["family"].values, "sales": P_test[j]})
     for j, d in enumerate(test_dates)],
    ignore_index=True,
)
submission = test.merge(long, on=["date", "store_nbr", "family"], how="left")[["id", "sales"]]
assert submission["sales"].notna().all()

SUB_PATH = os.path.join(ART_DIR, "submission_smoke.csv") if SMOKE else "../submission_04_hybrid.csv"
submission.to_csv(SUB_PATH, index=False)
print(f"Saved: {SUB_PATH}, {submission.shape[0]:,} rows, scheme = {FINAL_SCHEME}")
print(f"Total runtime: {(time.time() - _t0) / 60:.1f} min")
submission.head()

Saved: ../submission_04_hybrid.csv, 28,512 rows, scheme = hybrid_best
Total runtime: 1.2 min


,id,sales
0,3000888,3.687383
1,3000889,0.000000
2,3000890,5.439687
3,3000891,2290.032724
4,3000892,0.031468


## Conclusions

**Validation (3 honest windows).** The winner is `hybrid_best` (mean RMSLE **0.39039**): the `global_2016` base with 14 families switched to their per-family/blend variant, each switch confirmed on all 3 windows. The simple `blend_pf25_g16` (25% per-family + 75% global) is effectively tied (**0.39046**). Pure per-family models are the weakest of the models (0.39812): 33 small models overfit and miss cross-family patterns, so they help only inside a blend. The honest proxy of the deployed submission (`global_2017`) is the worst scheme of all (0.40123): the original `w_2017 = 1.0` in notebook 03 was an artifact of the leakage.

**Leaderboard.** `submission_04_hybrid.csv` (hybrid_best) scored **0.39243** vs **0.39109** for the old submission — the validation gain did not transfer. The reason is structural: in any honest pre-test window a model trained "from 2017" has never seen August, while the deployed `model_2017` is trained through Aug 15 and has the freshest two weeks of data. This freshness advantage is invisible to a validation that stops before the test period, so the 2016-vs-2017 choice cannot be made from these windows.

**What stands:** the leakage fix in notebook 03 (honest RMSLE ≈ 0.399 vs the leaked 0.293 — the public 0.391 lands where the honest estimate predicts); the 3-window methodology and the matrix engine; and the finding that per-family models help only inside blends.

**Lesson:** honest validation measures quality under its own information set. When the deployed model has richer (fresher) data, comparing schemes with different data recency does not transfer to the leaderboard.

## Выводы

**Валидация (3 честных окна).** Победил `hybrid_best` (среднее RMSLE **0.39039**): база `global_2016`, у которой 14 категорий переключены на свой per-family/бленд-вариант, и каждое переключение подтверждено на всех 3 окнах. Простой `blend_pf25_g16` (25% per-family + 75% global) практически неотличим (**0.39046**). Чистые per-family — самые слабые среди моделей (0.39812): 33 маленькие модели переобучаются и теряют межкатегорийные паттерны, поэтому полезны только внутри бленда. Честный аналог задеплоенного сабмита (`global_2017`) — худшая схема из всех (0.40123): исходный выбор `w_2017 = 1.0` в ноутбуке 03 был артефактом утечки.

**Лидерборд.** `submission_04_hybrid.csv` (hybrid_best) набрал **0.39243** против **0.39109** у старого сабмита — выигрыш с валидации не перенёсся. Причина структурная: в любом честном до-тестовом окне модель «с 2017» никогда не видела августа, а боевая `model_2017` обучена по 15 августа и владеет свежайшими двумя неделями данных. Это преимущество свежести невидимо валидации, которая заканчивается до тестового периода, поэтому выбор «2016 против 2017» по этим окнам сделать нельзя.

**Что остаётся в силе:** фикс утечки в ноутбуке 03 (честный RMSLE ≈ 0.399 против протёкшего 0.293 — публичный 0.391 ложится туда, куда указывает честная оценка); методология 3 окон и матричный движок; вывод, что per-family полезны только внутри бленда.

**Урок:** честная валидация измеряет качество при своём информационном наборе. Когда у боевой модели данные богаче (свежее), сравнение схем с разной свежестью данных на лидерборд не переносится.